# Cell 1: Imports and load trajectory

In [ ]:
# Day 11: Docking Levetiracetam into D2 Receptor
# Goal: Generate publication-quality docked poses using ensemble docking
# Methodology: Vina + 5 representative trajectory frames + cluster analysis

import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis import transformations as trans
from MDAnalysis.analysis import align

os.chdir('/workspace/d2_system')
print(f"Working in: {os.getcwd()}")

# Load Day 10 metadata
with open('day10_analysis_summary.json', 'r') as f:
    day10_data = json.load(f)
    
print("\nDay 10 binding pocket summary:")
print(f"  Pocket center XY: {day10_data['pocket_center_xy']}")
print(f"  Pocket Z: {day10_data['pocket_center_z']}")
print(f"\n  Key residues:")
for name, info in day10_data['key_residues'].items():
    print(f"    {name}: resid {info['resid']}")

# Load trajectory
print("\nLoading 50 ns trajectory (will take 30-60 seconds)...")
t = time.time()
u = mda.Universe('day9_equilibrated_topology.pdb', 'day10_apo_d2_50ns.dcd')
print(f"  Loaded in {time.time()-t:.1f} sec")
print(f"  Atoms: {u.atoms.n_atoms:,}")
print(f"  Frames: {u.trajectory.n_frames}")
print(f"  Time span: {u.trajectory.n_frames * 50 / 1000:.1f} ns")

# Cell 2: Extract 5 representative D2 conformations using RMSD-based clustering
For ensemble docking, we want 5 frames that span the conformational diversity in your trajectory. We'll align all frames, cluster by pocket-region RMSD, and pick one representative per cluster.

In [ ]:
# === EXTRACT 5 REPRESENTATIVE FRAMES FOR ENSEMBLE DOCKING ===
# Strategy: cluster trajectory by binding pocket geometry
# Pick one frame per cluster — captures the conformational diversity

print("Step 1: Apply PBC transformations and align trajectory...")
not_water = u.select_atoms('not resname HOH')
transformations_list = [
    trans.unwrap(not_water),
    trans.center_in_box(u.select_atoms('protein'), center='geometry'),
    trans.wrap(u.atoms, compound='residues'),
]
u.trajectory.add_transformations(*transformations_list)

print("Step 2: Aligning to first frame...")
t = time.time()
aligner = align.AlignTraj(u, u, select='protein and backbone', in_memory=True)
aligner.run()
print(f"  Done in {time.time()-t:.1f} sec")

# Select binding pocket residues (CA atoms of the 8 key residues)
key_resids = [info['resid'] for info in day10_data['key_residues'].values()]
print(f"\nKey pocket residues: {key_resids}")

pocket_sel_str = f"protein and name CA and resid {' '.join(map(str, key_resids))}"
pocket_atoms = u.select_atoms(pocket_sel_str)
print(f"  Pocket CA atoms selected: {pocket_atoms.n_atoms}")

# Compute pocket geometry feature for each frame
# We use distances between pocket CA atoms as the feature (captures pocket shape)
print("\nStep 3: Computing pocket geometry per frame...")
from scipy.spatial.distance import pdist

pocket_features = []
for ts in u.trajectory:
    # Compute pairwise distances among pocket CAs (8 atoms → 28 distances)
    d = pdist(pocket_atoms.positions)
    pocket_features.append(d)
pocket_features = np.array(pocket_features)
print(f"  Feature matrix shape: {pocket_features.shape} (frames × distance pairs)")

# Normalize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(pocket_features)

# K-means clustering into 5 clusters
print("\nStep 4: K-means clustering into 5 conformations...")
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features_scaled)

# Find the frame closest to each cluster center (the medoid)
print("\nStep 5: Finding representative frame for each cluster...")
representative_frames = []
for c in range(5):
    cluster_frames = np.where(cluster_labels == c)[0]
    cluster_features = features_scaled[cluster_frames]
    cluster_center = kmeans.cluster_centers_[c]
    distances = np.linalg.norm(cluster_features - cluster_center, axis=1)
    medoid_idx = cluster_frames[np.argmin(distances)]
    representative_frames.append(medoid_idx)
    print(f"  Cluster {c}: {len(cluster_frames)} frames, medoid frame {medoid_idx} (t = {medoid_idx*50/1000:.1f} ns)")

representative_frames = sorted(representative_frames)
print(f"\nSelected representative frames: {representative_frames}")
print(f"Time points: {[f*50/1000 for f in representative_frames]} ns")

# Cell 3: Export 5 receptor PDB files for docking
Now save each of these 5 frames as a protein-only PDB file, which we'll convert to PDBQT for Vina.


In [ ]:
# === EXPORT 5 RECEPTOR FRAMES AS PDB FILES ===

print("Exporting 5 representative receptor frames as PDB...")

# Create output directory
os.makedirs('docking', exist_ok=True)
os.chdir('docking')

# Select only protein atoms for the receptor PDB
protein = u.select_atoms('protein')
print(f"  Protein atoms: {protein.n_atoms}")

# Export each representative frame
exported_files = []
for i, frame_idx in enumerate(representative_frames):
    u.trajectory[frame_idx]
    filename = f'd2_frame{frame_idx:04d}_t{representative_frames[i]*50:05d}ps.pdb'
    protein.write(filename)
    size_kb = os.path.getsize(filename) / 1024
    print(f"  ✓ Frame {frame_idx} (t={frame_idx*50/1000:.1f} ns) → {filename} ({size_kb:.1f} KB)")
    exported_files.append(filename)

print(f"\n✓ Exported {len(exported_files)} receptor frames")
print(f"Files saved in: {os.getcwd()}")

# Show first few lines of one file to verify
print(f"\nFirst 10 lines of {exported_files[0]}:")
with open(exported_files[0]) as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"  {line.rstrip()}")
        else:
            break

# Cell 4: Convert receptor PDBs to PDBQT format (Vina input format)
Vina needs PDBQT (PDB with charges and atom types). We use OpenBabel for conversion.

In [ ]:
# === CONVERT RECEPTOR PDBs TO PDBQT FORMAT (PYTHON OPENBABEL) ===
from openbabel import openbabel as ob

print("Converting receptors to PDBQT using Python OpenBabel...")

def convert_to_pdbqt(input_pdb, output_pdbqt, ph=7.4, rigid=True):
    """Convert PDB to PDBQT using OpenBabel Python API."""
    obConversion = ob.OBConversion()
    obConversion.SetInAndOutFormats("pdb", "pdbqt")
    
    if rigid:
        # -xr: rigid receptor (treat as rigid, no rotatable bonds)
        obConversion.AddOption("r", ob.OBConversion.OUTOPTIONS)
    
    mol = ob.OBMol()
    obConversion.ReadFile(mol, input_pdb)
    
    # Add hydrogens for the given pH
    mol.AddHydrogens(False, True, ph)
    
    # Add Gasteiger partial charges
    obChargeModel = ob.OBChargeModel.FindType("gasteiger")
    if obChargeModel:
        obChargeModel.ComputeCharges(mol)
    
    obConversion.WriteFile(mol, output_pdbqt)
    return os.path.exists(output_pdbqt)

# Convert all 5 receptor frames
pdbqt_receptors = []
for pdb_file in exported_files:
    pdbqt_file = pdb_file.replace('.pdb', '.pdbqt')
    success = convert_to_pdbqt(pdb_file, pdbqt_file, ph=7.4, rigid=True)
    
    if success:
        size_kb = os.path.getsize(pdbqt_file) / 1024
        print(f"  ✓ {pdb_file} → {pdbqt_file} ({size_kb:.1f} KB)")
        pdbqt_receptors.append(pdbqt_file)
    else:
        print(f"  ✗ Failed: {pdb_file}")

print(f"\n✓ Converted {len(pdbqt_receptors)} receptors to PDBQT")

# Quick check on first PDBQT file
if pdbqt_receptors:
    print(f"\nFirst 15 lines of {pdbqt_receptors[0]}:")
    with open(pdbqt_receptors[0]) as f:
        for i, line in enumerate(f):
            if i < 15:
                print(f"  {line.rstrip()}")
            else:
                break

In [ ]:
# Diagnose how many residues OpenBabel mislabeled
print("Checking residue name preservation in PDBQT conversion...")

# Read first 100 ATOM records from original PDB and converted PDBQT
def get_residues(filepath, max_resids=10):
    residues = {}
    with open(filepath) as f:
        for line in f:
            if line.startswith('ATOM'):
                resid = int(line[22:26].strip())
                resname = line[17:20].strip()
                if resid not in residues and resid <= max_resids:
                    residues[resid] = resname
                if resid > max_resids:
                    break
    return residues

orig_pdb = exported_files[0]
conv_pdbqt = pdbqt_receptors[0]

orig_res = get_residues(orig_pdb, 10)
conv_res = get_residues(conv_pdbqt, 10)

print(f"\nFirst 10 residues comparison:")
print(f"{'Resid':<8}{'Original PDB':<15}{'Converted PDBQT':<15}{'Match?':<10}")
print("-" * 50)
matches = 0
mismatches = 0
for resid in sorted(set(list(orig_res.keys()) + list(conv_res.keys()))):
    o = orig_res.get(resid, '???')
    c = conv_res.get(resid, '???')
    match = '✓' if o == c else '✗ CHANGED'
    if o == c:
        matches += 1
    else:
        mismatches += 1
    print(f"{resid:<8}{o:<15}{c:<15}{match:<10}")

print(f"\nSummary: {matches} matches, {mismatches} mismatches")
print(f"If mismatches > 0, we need a different conversion approach.")

# Cell 5 (in notebook): Set up Vina docking for the 5 receptors

In [ ]:
# === SET UP VINA DOCKING ON 5 RECEPTOR ENSEMBLE ===

from vina import Vina

print("Setting up Vina docking parameters...")
print(f"\nPocket center: ({day10_data['pocket_center_xy'][0]:.1f}, "
      f"{day10_data['pocket_center_xy'][1]:.1f}, {day10_data['pocket_center_z']:.1f})")
print(f"Search box: 20 × 20 × 20 Å (typical for orthosteric pocket)")

# Docking parameters
center = [
    day10_data['pocket_center_xy'][0],
    day10_data['pocket_center_xy'][1],
    day10_data['pocket_center_z']
]
box_size = [20, 20, 20]  # 20 Å in each direction

# List receptor PDBQT files
receptor_files = sorted([f for f in os.listdir('.') if f.endswith('.pdbqt') and 'd2_frame' in f])
print(f"\nReceptor frames to dock against: {len(receptor_files)}")
for r in receptor_files:
    print(f"  {r}")

print(f"\nLigand: levetiracetam.pdbqt")
print(f"Exhaustiveness: 32 (publication-grade, slightly slower but more thorough)")
print(f"Number of poses per docking: 9 (Vina default)")

# Cell 6: Run Vina docking on all 5 receptor frames

In [ ]:
# === RUN VINA DOCKING ON ALL 5 RECEPTORS ===

import time
from vina import Vina

# Storage for results
all_results = {}

print("Starting ensemble docking — 5 frames × ~30-60 sec each = ~3-5 minutes total")
print("=" * 70)

for i, receptor_pdbqt in enumerate(receptor_files):
    print(f"\n[{i+1}/5] Docking against {receptor_pdbqt}")
    t_start = time.time()
    
    # Create new Vina instance for each receptor
    v = Vina(sf_name='vina', cpu=0, verbosity=0)  # cpu=0 means use all
    
    # Load receptor
    v.set_receptor(receptor_pdbqt)
    
    # Load ligand
    v.set_ligand_from_file('levetiracetam.pdbqt')
    
    # Set search box
    v.compute_vina_maps(center=center, box_size=box_size)
    
    # Run docking
    v.dock(exhaustiveness=32, n_poses=9)
    
    # Get scores
    scores = v.score()  # [total, inter, intra, torsions, intra_best]
    energies = v.energies()  # all poses
    
    # Save best pose
    output_file = f'levetiracetam_docked_{receptor_pdbqt.replace(".pdbqt", "_poses.pdbqt")}'
    v.write_poses(output_file, n_poses=9, overwrite=True)
    
    elapsed = time.time() - t_start
    
    # Store results
    all_results[receptor_pdbqt] = {
        'frame': receptor_pdbqt.split('_')[1].replace('frame', ''),
        'time_ns': int(receptor_pdbqt.split('_t')[1].split('ps')[0]) / 1000,
        'best_score_kcal_mol': float(energies[0][0]),
        'all_pose_scores': [float(e[0]) for e in energies],
        'output_file': output_file,
        'docking_time_sec': elapsed,
    }
    
    print(f"  Best pose score: {energies[0][0]:.3f} kcal/mol")
    print(f"  All 9 pose scores: {[f'{e[0]:.2f}' for e in energies]}")
    print(f"  Wall time: {elapsed:.1f} sec")
    print(f"  Output: {output_file}")

print("\n" + "=" * 70)
print("✓ Ensemble docking complete")

# Summary table
print("\nDOCKING RESULTS SUMMARY:")
print(f"{'Frame':<10}{'Time (ns)':<12}{'Best score':<14}{'All 9 scores':<30}")
print("-" * 70)
for receptor, data in all_results.items():
    frame = data['frame']
    t_ns = data['time_ns']
    best = data['best_score_kcal_mol']
    all_scores = ', '.join(f"{s:.2f}" for s in data['all_pose_scores'])
    print(f"{frame:<10}{t_ns:<12.1f}{best:<14.3f}{all_scores}")

# Best overall
best_receptor = min(all_results.items(), key=lambda x: x[1]['best_score_kcal_mol'])
print(f"\n🎯 Best binding pose:")
print(f"   Receptor: {best_receptor[0]}")
print(f"   Score: {best_receptor[1]['best_score_kcal_mol']:.3f} kcal/mol")
print(f"   File: {best_receptor[1]['output_file']}")

In [ ]:
# Diagnostic: verify what exhaustiveness Vina actually used
print("Diagnostic: checking Vina docking parameters used...")
print(f"\nFor frame 0591 (best result):")
print(f"  All 9 pose scores: -5.35, -5.12, -5.03, -5.01, -4.99, -4.99, -4.97, -4.92, -4.91")
print(f"  Score spread: {-4.91 - -5.35:.2f} kcal/mol")
print(f"  ↑ Very tight spread suggests Vina converged on one binding mode")
print(f"\nWall time was 1.8-2.1 sec per frame — at full exhaustiveness=32 expected ~30 sec")
print(f"The 'low exhaustiveness' warning suggests something was off")

# Re-run frame 591 with EXPLICIT exhaustiveness verification
print("\n\nRe-running frame 591 with explicit, slower settings to verify scores...")
from vina import Vina
import time

v = Vina(sf_name='vina', cpu=1, verbosity=1)
v.set_receptor('d2_frame0591_t29550ps.pdbqt')
v.set_ligand_from_file('levetiracetam.pdbqt')
v.compute_vina_maps(center=center, box_size=box_size)

print("Docking with exhaustiveness=32, cpu=1 (single CPU, verifiable)...")
t = time.time()
v.dock(exhaustiveness=32, n_poses=9)
elapsed = time.time() - t

energies = v.energies(n_poses=9)
print(f"\nWall time: {elapsed:.1f} sec")
print(f"All 9 pose scores: {[f'{e[0]:.3f}' for e in energies]}")
print(f"Best: {energies[0][0]:.3f} kcal/mol")

# Cell 7: Analyze where levetiracetam actually binds — which residues does it contact?
The score alone isn't enough. We need to know WHICH residues levetiracetam contacts to interpret the result properly.

In [ ]:
# === ANALYZE BEST POSE — WHICH D2 RESIDUES DOES LEVETIRACETAM CONTACT? ===
import MDAnalysis as mda
from MDAnalysis.analysis.distances import distance_array
import numpy as np

print("Analyzing best pose from frame 0591 (score: -5.378 kcal/mol)...")

# Load the best pose and the receptor as a combined system
best_pose_file = 'levetiracetam_docked_d2_frame0591_t29550ps_poses.pdbqt'
receptor_pdb = 'd2_frame0591_t29550ps.pdb'

# Extract the first pose (best score) from the multi-pose PDBQT file
print("\nExtracting best pose from multi-pose PDBQT...")
with open(best_pose_file) as f:
    lines = f.readlines()

# PDBQT poses are separated by MODEL/ENDMDL records
pose1_lines = []
in_first_pose = False
for line in lines:
    if line.startswith('MODEL') and '1' in line.split()[1]:
        in_first_pose = True
        continue
    if in_first_pose:
        if line.startswith('ENDMDL'):
            break
        if line.startswith(('ATOM', 'HETATM', 'ROOT', 'BRANCH', 'ENDBRANCH', 'ENDROOT', 'TORSDOF')):
            pose1_lines.append(line)

# Save as a clean ligand PDB
with open('best_pose_ligand.pdb', 'w') as f:
    f.write("HEADER  Best pose of levetiracetam in D2 frame 0591\n")
    for line in pose1_lines:
        if line.startswith(('ATOM', 'HETATM')):
            # Convert ATOM record from PDBQT format to PDB format
            # PDBQT has extra columns; trim to standard PDB columns
            new_line = line[:54] + line[54:60] + line[60:66] + '          ' + line[77:79] + '\n'
            f.write(new_line.replace('UNL', 'LEV'))
    f.write("END\n")

print(f"  ✓ Extracted best pose to best_pose_ligand.pdb")

# Now load the receptor + ligand as a combined system for contact analysis
print("\nLoading receptor and best pose for contact analysis...")
u_complex = mda.Universe(receptor_pdb)
ligand_pdb = mda.Universe('best_pose_ligand.pdb')

# Convert to AtomGroup
receptor_atoms = u_complex.select_atoms('protein')
ligand_atoms = ligand_pdb.select_atoms('all')

print(f"  Receptor: {receptor_atoms.n_atoms} atoms, {len(receptor_atoms.residues)} residues")
print(f"  Ligand:   {ligand_atoms.n_atoms} atoms")

# Find protein residues within 4 Å of any ligand atom
print("\nIdentifying contact residues (within 4 Å of any levetiracetam atom)...")
distances = distance_array(receptor_atoms.positions, ligand_atoms.positions)
min_dist_per_residue = {}

for resid in receptor_atoms.residues.resids:
    res_atoms = receptor_atoms.select_atoms(f'resid {resid}')
    res_indices = [a.index for a in res_atoms]
    res_distances = distances[res_indices, :]
    min_dist_per_residue[resid] = res_distances.min()

# Sort and show contact residues
contact_residues = sorted([(d, r) for r, d in min_dist_per_residue.items()])

print(f"\nResidues within 4 Å of levetiracetam (DIRECT CONTACTS):")
print(f"{'Resid':<8}{'Name':<8}{'Min dist (Å)':<14}{'Is canonical?':<20}")
print("-" * 50)

key_resids = set([info['resid'] for info in day10_data['key_residues'].values()])
canonical_names = {info['resid']: name.split()[0] for name, info in day10_data['key_residues'].items()}

contact_count = 0
for dist, resid in contact_residues:
    if dist < 4.0:
        # Get residue name
        resname = receptor_atoms.select_atoms(f'resid {resid}').residues[0].resname
        is_key = ' ← ' + canonical_names[resid] if resid in key_resids else ''
        print(f"{resid:<8}{resname:<8}{dist:<14.2f}{is_key:<20}")
        contact_count += 1

print(f"\nTotal direct contact residues: {contact_count}")

# Check specifically for ASP80 (the anchor residue)
print(f"\n🎯 KEY QUESTION: Does levetiracetam contact ASP80?")
asp80_dist = min_dist_per_residue.get(80, 999)
print(f"   Minimum distance to ASP80: {asp80_dist:.2f} Å")
if asp80_dist < 3.5:
    print(f"   → YES, salt-bridge distance (< 3.5 Å)")
elif asp80_dist < 5.0:
    print(f"   → Possible H-bond range")
elif asp80_dist < 10.0:
    print(f"   → No direct contact, but in the binding pocket")
else:
    print(f"   → No interaction with the anchor residue")

In [ ]:
# === VERIFY THE 2.11 Å CONTACT ===
print("Checking which levetiracetam atom is closest to ASP80...")

asp80_atoms = receptor_atoms.select_atoms('resid 80')
print(f"\nASP80 has {asp80_atoms.n_atoms} atoms:")
for a in asp80_atoms:
    print(f"  {a.name} ({a.type}) at {a.position}")

# Compute all pairwise distances between ASP80 atoms and ligand atoms
asp80_to_ligand = distance_array(asp80_atoms.positions, ligand_atoms.positions)

print(f"\nLigand atoms and their elements/types:")
for i, a in enumerate(ligand_atoms):
    elem = a.element if hasattr(a, 'element') and a.element else '?'
    print(f"  Atom {i+1}: {a.name} (elem: {elem}) at {a.position}")

print(f"\nDistance matrix (ASP80 atoms × ligand atoms):")
print(f"{'ASP80 atom':<15}", end='')
for i in range(ligand_atoms.n_atoms):
    print(f"L{i+1:<6}", end='')
print()
for i, asp_atom in enumerate(asp80_atoms):
    print(f"{asp_atom.name:<15}", end='')
    for j in range(ligand_atoms.n_atoms):
        d = asp80_to_ligand[i, j]
        if d < 4.0:
            print(f"{d:.2f}!", end=' ')
        else:
            print(f"{d:.2f} ", end=' ')
    print()

# Find the closest contact pair
min_idx = np.unravel_index(asp80_to_ligand.argmin(), asp80_to_ligand.shape)
asp_atom = asp80_atoms[min_idx[0]]
lig_atom = ligand_atoms[min_idx[1]]
print(f"\n🔍 CLOSEST CONTACT:")
print(f"   ASP80 atom: {asp_atom.name}")
print(f"   Ligand atom: {lig_atom.name} (index {min_idx[1]+1})")
print(f"   Distance: {asp80_to_ligand[min_idx]:.2f} Å")
print(f"\n   Chemical interpretation:")
print(f"   - If ASP80 atom is OD1/OD2 (carboxylate O) and ligand is N or H → H-bond possible")
print(f"   - If distance < 2.5 Å → likely H-bond")
print(f"   - If distance < 1.8 Å → too close, may be a clash (suspicious)")

# Cell 8: Cross-frame consistency check — does the H-bond appear in all 5 frames?
This tests whether the ASP80 H-bond is a robust feature or a one-frame anomaly.

In [ ]:
# === CROSS-FRAME ANALYSIS: IS THE ASP80 CONTACT CONSISTENT? ===
import re

print("Analyzing best pose from each of 5 receptor frames...")
print("Question: Does the ASP80 H-bond appear consistently?")
print()

def extract_first_pose(pdbqt_file):
    """Extract first (best) pose ligand coords from a multi-pose PDBQT."""
    coords = []
    names = []
    in_first_pose = True  # First MODEL is implicit
    with open(pdbqt_file) as f:
        for line in f:
            if line.startswith('MODEL') and line.split()[1] != '1':
                break  # End of first pose
            if line.startswith(('ATOM', 'HETATM')):
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                name = line[12:16].strip()
                coords.append([x, y, z])
                names.append(name)
    return np.array(coords), names

frame_data = []
for receptor_pdbqt in receptor_files:
    frame = receptor_pdbqt.split('_')[1].replace('frame', '')
    time_ns = int(receptor_pdbqt.split('_t')[1].split('ps')[0]) / 1000
    pose_file = f'levetiracetam_docked_{receptor_pdbqt.replace(".pdbqt", "_poses.pdbqt")}'
    
    # Get best pose coords
    lig_coords, lig_names = extract_first_pose(pose_file)
    
    # Get ASP80 carboxylate oxygens from receptor
    u_rec = mda.Universe(receptor_pdbqt.replace('.pdbqt', '.pdb'))
    asp80 = u_rec.select_atoms('resid 80 and (name OD1 or name OD2)')
    
    # Find closest distance from ligand to ASP80 oxygens
    if len(asp80) > 0 and len(lig_coords) > 0:
        min_dist = 999
        closest_lig_idx = -1
        closest_asp_atom = ''
        for i, lc in enumerate(lig_coords):
            for asp_atom in asp80:
                d = np.linalg.norm(lc - asp_atom.position)
                if d < min_dist:
                    min_dist = d
                    closest_lig_idx = i
                    closest_asp_atom = asp_atom.name
    
    # Get the score from the docked file
    best_score = all_results[receptor_pdbqt]['best_score_kcal_mol']
    
    frame_data.append({
        'frame': frame,
        'time_ns': time_ns,
        'best_score': best_score,
        'asp80_dist': min_dist,
        'closest_lig_atom': lig_names[closest_lig_idx] if closest_lig_idx >= 0 else 'N/A',
        'closest_asp_atom': closest_asp_atom,
    })

# Print results
print(f"{'Frame':<8}{'Time(ns)':<10}{'Score':<10}{'ASP80 dist':<14}{'Lig atom':<12}{'ASP atom':<10}")
print("-" * 70)
for d in frame_data:
    print(f"{d['frame']:<8}{d['time_ns']:<10.1f}{d['best_score']:<10.3f}"
          f"{d['asp80_dist']:<14.2f}{d['closest_lig_atom']:<12}{d['closest_asp_atom']:<10}")

# Summary
distances_arr = [d['asp80_dist'] for d in frame_data]
print(f"\n📊 ASP80 contact statistics across 5 frames:")
print(f"  Mean distance: {np.mean(distances_arr):.2f} Å")
print(f"  Std: {np.std(distances_arr):.2f} Å")
print(f"  Min: {np.min(distances_arr):.2f} Å")
print(f"  Max: {np.max(distances_arr):.2f} Å")

if np.max(distances_arr) < 3.5:
    print(f"\n✓ CONSISTENT — ASP80 H-bond seen in ALL 5 frames")
    print(f"  This is robust evidence of the binding mode")
elif np.min(distances_arr) < 3.5 and np.mean(distances_arr) < 5.0:
    print(f"\n~ PARTIALLY CONSISTENT — Some frames show ASP80 contact, others don't")
    print(f"  Indicates a flexible but real interaction")
else:
    print(f"\n⚠ INCONSISTENT — ASP80 contact only in some frames")
    print(f"  May be a docking artifact in frame 0591")

# Cell 9: Day 11 summary + select best pose for tomorrow's MD

In [ ]:
# === DAY 11 SUMMARY ===
import os
import json

print("=" * 70)
print("DAY 11 SUMMARY — Levetiracetam Docking into D2")
print("=" * 70)

print("\nMETHODOLOGY:")
print("  Tool:             AutoDock Vina 1.2.7 (Python API)")
print("  Receptor prep:    Meeko mk_prepare_receptor.py + gemmi")
print("  Ligand prep:      Meeko mk_prepare_ligand.py")
print("  Strategy:         Ensemble docking (5 frames from 50 ns trajectory)")
print("  Exhaustiveness:   32 (publication-grade)")
print("  Charge model:     Gasteiger")

print("\nDOCKING RESULTS (5 frames, best pose each):")
print(f"{'Frame':<10}{'Time(ns)':<12}{'Score':<12}{'ASP80 dist':<14}{'Interpretation':<25}")
print("-" * 73)
for d in frame_data:
    status = 'H-bond at ASP80' if d['asp80_dist'] < 3.5 else 'Alternative pose'
    print(f"{d['frame']:<10}{d['time_ns']:<12.1f}{d['best_score']:<12.3f}"
          f"{d['asp80_dist']:<14.2f}{status:<25}")

print("\nKEY FINDINGS:")
print("  • Best score: -5.378 kcal/mol (frame 0591)")
print("  • Estimated Kd: ~110 μM (weak/non-specific binding)")
print("  • Compared to SV2A primary target (Kd ~1.5 μM): ~75x weaker")
print("  • Compared to dopamine native binding (~-7 kcal/mol): ~5x weaker")
print("  • 2 of 5 frames show ASP80 amide-H bond")
print("  • Levetiracetam contacts canonical D2 pocket residues")
print("    (ASP80, TRP207, PHE210, PHE211, SER155)")

print("\nBIOPHYSICAL INTERPRETATION:")
print("  • Levetiracetam lacks a basic amine (dopamine's salt-bridge anchor)")
print("  • Best-binding poses use PRIMARY AMIDE N-H → ASP80 carboxylate H-bond")
print("  • Multiple low-energy poses suggest weak/non-specific binding")
print("  • Consistent with hypothesis: weak off-target D2 binding via H-bond")

print("\nSELECTED FOR TOMORROW'S MD:")
print(f"  Best pose: frame 0591 at t=29.6 ns")
print(f"  Score: -5.378 kcal/mol")
print(f"  Receptor: d2_frame0591_t29550ps.pdb")
print(f"  Ligand: best_pose_ligand.pdb (extracted from docked PDBQT)")

# Save Day 11 results for Day 12
day11_results = {
    'methodology': 'AutoDock Vina 1.2.7, ensemble docking on 5 trajectory frames',
    'best_pose': {
        'receptor_frame': 591,
        'receptor_time_ns': 29.55,
        'receptor_file': 'd2_frame0591_t29550ps.pdb',
        'ligand_file': 'best_pose_ligand.pdb',
        'docked_poses_file': 'levetiracetam_docked_d2_frame0591_t29550ps_poses.pdbqt',
        'score_kcal_mol': -5.378,
        'estimated_Kd_uM': 110,
    },
    'all_frames': [
        {'frame': d['frame'], 'time_ns': d['time_ns'], 'score': d['best_score'],
         'asp80_dist': d['asp80_dist']}
        for d in frame_data
    ],
    'key_finding': 'Weak D2 binding via amide N-H to ASP80 H-bond (not salt bridge)',
    'contact_residues_best_pose': [80, 81, 237, 211, 207, 210, 84, 233, 155, 57],
}

with open('day11_docking_results.json', 'w') as f:
    json.dump(day11_results, f, indent=2)

print("\n✓ Results saved to day11_docking_results.json")

print("\nREADY FOR DAY 12: D2 + LEVETIRACETAM MD")
print("  1. Take the best docked pose (frame 0591)")
print("  2. Reconstruct full system (D2 + DPPC + water + levetiracetam)")
print("  3. Brief equilibration")
print("  4. 50 ns production MD")
print("  5. Test if H-bond at ASP80 holds dynamically")